# 🐍 Aula 07 - API de DataFrame do PySpark

## Objetivo

Esta aula demonstra como usar a **API de DataFrame do PySpark** para realizar as mesmas transformações que fizemos na Aula 06 com SQL, mas agora usando programação imperativa com Python.

### O que vamos aprender:

1. **Leitura de CSVs** com schema inference vs schema explícito
2. **Transformações básicas**: `select`, `filter`, `withColumn`, `orderBy`
3. **Joins** entre DataFrames
4. **Agregações** com `groupBy` e `agg`
5. **Transformations vs Actions** (lazy evaluation)

---


## 📚 Parte 1: Leitura de Dados com PySpark

### 1.1. SparkSession (já existe no Databricks)

No Databricks, a `SparkSession` já está disponível como `spark`. Se estiver rodando localmente, você precisaria criar:


In [ ]:
# No Databricks, a SparkSession já existe
# Se estiver rodando localmente, descomente as linhas abaixo:

# from pyspark.sql import SparkSession
# spark = SparkSession.builder \
#     .appName("aula_07_pyspark") \
#     .master("local[*]") \
#     .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"SparkSession criada: {spark}")


### 1.2. Leitura com Schema Inference

A forma mais simples de ler dados é usando **schema inference**, onde o Spark tenta adivinhar os tipos automaticamente.


In [ ]:
# Leitura com schema inference usando os volumes do Databricks
customers_df_inferred = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")  # << Schema inference ativado
    .load("/Volumes/smart_claims_dev/00_landing/sql_server/customers/")
)

print("\n=== Schema Inferido ===")
customers_df_inferred.printSchema()

print("\n=== Primeiras 5 linhas ===")
customers_df_inferred.show(5, truncate=False)


### 1.3. Definindo Schemas Explícitos

Em produção, é **boa prática definir o schema explicitamente** para:
- Evitar custos de leitura extra (schema inference precisa ler os dados)
- Garantir tipos corretos (evitar erros de inferência)
- Melhor performance


In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType
)

# Schema explícito para customers
customers_schema = StructType([
    StructField("customer_id", DoubleType(), True),
    StructField("date_of_birth", StringType(), True),  # Vamos tratar como string primeiro
    StructField("borough", StringType(), True),
    StructField("neighborhood", StringType(), True),
    StructField("zip_code", StringType(), True),
    StructField("name", StringType(), True),
])

# Schema explícito para policies
policies_schema = StructType([
    StructField("POLICY_NO", StringType(), True),
    StructField("CUST_ID", DoubleType(), True),
    StructField("POLICYTYPE", StringType(), True),
    StructField("POL_ISSUE_DATE", StringType(), True),
    StructField("POL_EFF_DATE", StringType(), True),
    StructField("POL_EXPIRY_DATE", StringType(), True),
    StructField("MAKE", StringType(), True),
    StructField("MODEL", StringType(), True),
    StructField("MODEL_YEAR", DoubleType(), True),
    StructField("CHASSIS_NO", StringType(), True),
    StructField("USE_OF_VEHICLE", StringType(), True),
    StructField("PRODUCT", StringType(), True),
    StructField("SUM_INSURED", DoubleType(), True),
    StructField("PREMIUM", DoubleType(), True),
    StructField("DEDUCTABLE", IntegerType(), True),
])

# Schema explícito para claims
claims_schema = StructType([
    StructField("claim_no", StringType(), True),
    StructField("policy_no", StringType(), True),
    StructField("claim_date", StringType(), True),
    StructField("months_as_customer", IntegerType(), True),
    StructField("injury", IntegerType(), True),
    StructField("property", IntegerType(), True),
    StructField("vehicle", IntegerType(), True),
    StructField("total", IntegerType(), True),
    StructField("collision_type", StringType(), True),
    StructField("number_of_vehicles_involved", IntegerType(), True),
    StructField("age", DoubleType(), True),
    StructField("insured_relationship", StringType(), True),
    StructField("license_issue_date", StringType(), True),
    StructField("date", StringType(), True),
    StructField("hour", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("severity", StringType(), True),
    StructField("number_of_witnesses", IntegerType(), True),
    StructField("suspicious_activity", StringType(), True),
])

print("Schemas definidos com sucesso!")


### 1.4. Leitura com Schema Explícito

Agora vamos ler os dados usando os schemas definidos e os volumes do Databricks:


In [ ]:
# Leitura com schema explícito usando os volumes do Databricks
customers_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customers_schema)  # << Schema explícito
    .load("/Volumes/smart_claims_dev/00_landing/sql_server/customers/")
)

policies_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(policies_schema)
    .load("/Volumes/smart_claims_dev/00_landing/sql_server/policies/")
)

claims_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(claims_schema)
    .load("/Volumes/smart_claims_dev/00_landing/sql_server/claims/")
)

print("\n=== Schema Explícito - Customers ===")
customers_df.printSchema()

print("\n=== Contagem de registros ===")
print(f"Customers: {customers_df.count()}")
print(f"Policies: {policies_df.count()}")
print(f"Claims: {claims_df.count()}")


In [ ]:
from pyspark.sql import functions as F

# Selecionar colunas específicas
customers_bronze = customers_df.select(
    "customer_id",
    "name",
    "date_of_birth",
    "borough",
    "neighborhood",
    "zip_code"
)

# Renomear colunas usando alias
customers_bronze_renamed = customers_df.select(
    F.col("customer_id").alias("customer_id"),
    F.col("name").alias("customer_name"),  # Renomeando
    "date_of_birth",
    "borough",
    "neighborhood",
    "zip_code"
)

print("\n=== Customers Bronze (renomeado) ===")
customers_bronze_renamed.show(5, truncate=False)


### 2.2. Criando Novas Colunas com `withColumn`

O método `withColumn` permite criar ou modificar colunas:


In [ ]:
# Adicionar coluna de timestamp de processamento
customers_bronze_final = (
    customers_bronze_renamed
    .withColumn("processed_at", F.current_timestamp())
)

print("\n=== Customers Bronze com timestamp ===")
customers_bronze_final.show(5, truncate=False)
customers_bronze_final.printSchema()


### 2.3. Filtros com `filter` ou `where`

Ambos `filter` e `where` fazem a mesma coisa (são aliases):


In [ ]:
# Filtrar customers de Manhattan
customers_manhattan = customers_bronze_final.filter(
    F.col("borough") == "MANHATTAN"
)

# Ou usando where (mesma coisa)
customers_brooklyn = customers_bronze_final.where(
    F.col("borough") == "BROOKLYN"
)

print("\n=== Customers de Manhattan ===")
print(f"Total: {customers_manhattan.count()}")
customers_manhattan.show(5)

print("\n=== Customers de Brooklyn ===")
print(f"Total: {customers_brooklyn.count()}")


### 2.4. Ordenação com `orderBy`

Ordenar dados usando uma ou mais colunas:


In [ ]:
# Ordenar por customer_id
customers_ordered = customers_bronze_final.orderBy("customer_id")

# Ordenar descendente
customers_ordered_desc = customers_bronze_final.orderBy(
    F.col("customer_id").desc()
)

print("\n=== Primeiros 5 customers (ordenado ASC) ===")
customers_ordered.show(5)

print("\n=== Últimos 5 customers (ordenado DESC) ===")
customers_ordered_desc.show(5)


---

## ✨ Parte 3: Camada Silver - Joins e Enriquecimento

### 3.1. Preparando os DataFrames Bronze

Vamos preparar os DataFrames bronze antes de fazer os joins:


In [ ]:
# Preparar bronze de customers
customers_bronze = (
    customers_df
    .select(
        F.col("customer_id").cast("double"),
        "date_of_birth",
        "borough",
        "neighborhood",
        "zip_code",
        F.col("name").alias("customer_name")
    )
)

# Preparar bronze de policies
policies_bronze = (
    policies_df
    .select(
        F.col("POLICY_NO").cast("string").alias("policy_no"),
        F.col("CUST_ID").cast("double").alias("cust_id"),
        "POLICYTYPE",
        "POL_ISSUE_DATE",
        "POL_EFF_DATE",
        "POL_EXPIRY_DATE",
        "MAKE",
        "MODEL",
        "MODEL_YEAR",
        "CHASSIS_NO",
        "USE_OF_VEHICLE",
        "PRODUCT",
        "SUM_INSURED",
        "PREMIUM",
        "DEDUCTABLE"
    )
)

# Preparar bronze de claims
claims_bronze = (
    claims_df
    .select(
        "claim_no",
        F.col("policy_no").cast("string"),
        "claim_date",
        "months_as_customer",
        "injury",
        "property",
        "vehicle",
        F.col("total").alias("claim_total"),
        "collision_type",
        "number_of_vehicles_involved",
        "age",
        "insured_relationship",
        "license_issue_date",
        "date",
        "hour",
        "type",
        "severity",
        "number_of_witnesses",
        "suspicious_activity"
    )
)

print("\n=== Schema - Claims Bronze ===")
claims_bronze.printSchema()

print("\n=== Amostra - Claims Bronze ===")
claims_bronze.show(3, truncate=False)


In [ ]:
# Join Claims com Policies
claims_with_policies = (
    claims_bronze
    .join(
        policies_bronze,
        on=claims_bronze["policy_no"] == policies_bronze["policy_no"],
        how="inner"  # INNER JOIN
    )
)

print("\n=== Schema após join Claims + Policies ===")
claims_with_policies.printSchema()

print("\n=== Contagem de registros ===")
print(f"Claims originais: {claims_bronze.count()}")
print(f"Claims com policies: {claims_with_policies.count()}")

print("\n=== Amostra do join ===")
claims_with_policies.select(
    "claim_no",
    "policy_no",
    "claim_total",
    "MAKE",
    "MODEL",
    "cust_id"
).show(5, truncate=False)


### 3.3. Join Final: Claims + Policies + Customers

Agora vamos enriquecer com dados de customers:


In [ ]:
# Join final: Claims + Policies + Customers
claims_enriched = (
    claims_with_policies
    .join(
        customers_bronze,
        on=claims_with_policies["cust_id"] == customers_bronze["customer_id"],
        how="inner"
    )
    .withColumn("processed_at", F.current_timestamp())
)

print("\n=== Schema - Claims Enriched (Silver) ===")
claims_enriched.printSchema()

print("\n=== Contagem final ===")
print(f"Registros enriquecidos: {claims_enriched.count()}")

print("\n=== Amostra do resultado final ===")
claims_enriched.select(
    "claim_no",
    "policy_no",
    "claim_total",
    "MAKE",
    "MODEL",
    "customer_name",
    "borough",
    "neighborhood"
).show(5, truncate=False)


### 3.4. Selecionando Colunas Finais (Equivalente ao SQL da Aula 06)

Vamos selecionar apenas as colunas que queremos na camada Silver, igual ao SQL:


In [ ]:
# Selecionar colunas finais (equivalente ao SELECT do SQL)
claims_enriched_final = claims_enriched.select(
    # Colunas de claims
    "claim_no",
    "policy_no",
    "claim_date",
    "months_as_customer",
    "injury",
    "property",
    "vehicle",
    "claim_total",
    "collision_type",
    "number_of_vehicles_involved",
    "age",
    "insured_relationship",
    "license_issue_date",
    "date",
    "hour",
    "type",
    "severity",
    "number_of_witnesses",
    "suspicious_activity",
    # Colunas de policies
    "cust_id",
    "POLICYTYPE",
    "POL_ISSUE_DATE",
    "POL_EFF_DATE",
    "POL_EXPIRY_DATE",
    "MAKE",
    "MODEL",
    "MODEL_YEAR",
    "CHASSIS_NO",
    "USE_OF_VEHICLE",
    "PRODUCT",
    "SUM_INSURED",
    "PREMIUM",
    "DEDUCTABLE",
    # Colunas de customers
    "customer_id",
    "date_of_birth",
    "borough",
    "neighborhood",
    "zip_code",
    "customer_name",
    # Timestamp de processamento
    "processed_at"
)

print("\n=== Claims Enriched Final (Silver Layer) ===")
claims_enriched_final.show(3, truncate=False)

print("\n=== Schema Final ===")
claims_enriched_final.printSchema()


In [ ]:
# Métricas agregadas (equivalente ao SQL da Aula 06)
claims_metrics = (
    claims_enriched_final
    .agg(
        F.count("*").alias("total_enriched_records"),
        F.countDistinct("claim_no").alias("unique_claims"),
        F.countDistinct("policy_no").alias("unique_policies"),
        F.countDistinct("customer_id").alias("unique_customers")
    )
    .withColumn(
        "result_message",
        F.concat(
            F.lit("Join concluido. Total de registros enriquecidos: "),
            F.col("total_enriched_records").cast("string")
        )
    )
    .withColumn("calculated_at", F.current_timestamp())
)

print("\n=== Métricas Agregadas (Gold Layer) ===")
claims_metrics.show(truncate=False)


### 4.2. Agregações por Grupo (Exemplos Adicionais)

Vamos criar algumas métricas adicionais agrupadas por diferentes dimensões:


In [ ]:
# Métricas por borough
metrics_by_borough = (
    claims_enriched_final
    .groupBy("borough")
    .agg(
        F.count("*").alias("total_claims"),
        F.avg("claim_total").alias("avg_claim_total"),
        F.sum("claim_total").alias("sum_claim_total"),
        F.max("claim_total").alias("max_claim_total")
    )
    .orderBy(F.col("total_claims").desc())
)

print("\n=== Métricas por Borough ===")
metrics_by_borough.show(truncate=False)


In [ ]:
# Métricas por tipo de colisão
metrics_by_collision = (
    claims_enriched_final
    .groupBy("collision_type")
    .agg(
        F.count("*").alias("total_claims"),
        F.avg("claim_total").alias("avg_claim_total"),
        F.countDistinct("policy_no").alias("unique_policies")
    )
    .orderBy(F.col("total_claims").desc())
)

print("\n=== Métricas por Tipo de Colisão ===")
metrics_by_collision.show(truncate=False)


In [ ]:
# Métricas por severidade
metrics_by_severity = (
    claims_enriched_final
    .groupBy("severity")
    .agg(
        F.count("*").alias("total_claims"),
        F.avg("claim_total").alias("avg_claim_total"),
        F.sum("claim_total").alias("total_claim_amount")
    )
    .orderBy(F.col("total_claim_amount").desc())
)

print("\n=== Métricas por Severidade ===")
metrics_by_severity.show(truncate=False)


---

## 🎓 Parte 5: Conceitos Importantes

### 5.1. Transformations vs Actions

**Transformations** são operações **lazy** (preguiçosas) - elas não executam imediatamente:
- `select`, `filter`, `join`, `groupBy`, `withColumn`, etc.

**Actions** são operações **eager** (ansiosas) - elas executam imediatamente:
- `show()`, `count()`, `collect()`, `write()`, etc.

Vamos demonstrar:


In [ ]:
# Isso é uma TRANSFORMATION (não executa ainda)
transformation_example = claims_df.filter(F.col("total") > 10000)

print("\n=== Transformation criada (ainda não executou) ===")
print(f"Tipo: {type(transformation_example)}")
print("\nO Spark ainda não leu os dados nem aplicou o filtro!")
print("\nAgora vamos executar uma ACTION:")

# Isso é uma ACTION (executa agora)
count_result = transformation_example.count()
print(f"\nTotal de claims > 10000: {count_result}")
print("\nAgora sim o Spark executou a transformation + action!")


### 5.2. Visualizando o Plano de Execução

Podemos ver o plano de execução que o Spark criou:


In [ ]:
# Ver o plano de execução (explain)
print("\n=== Plano de Execução (Explain) ===")
claims_enriched_final.explain(True)

print("\n=== Plano de Execução Simplificado ===")
claims_enriched_final.explain()


### 5.3. Salvando Resultados (Opcional)

Podemos salvar os DataFrames em diferentes formatos:


In [ ]:
# Exemplo de como salvar (comentado para não executar acidentalmente)
# claims_enriched_final.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("smart_claims_dev.02_silver.claims_enriched_pyspark")

# Ou salvar como Parquet
# claims_enriched_final.write \
#     .format("parquet") \
#     .mode("overwrite") \
#     .save("/path/to/output/claims_enriched")

print("\n=== Exemplos de salvamento (comentados) ===")
print("Descomente as linhas acima para salvar os resultados!")


---

## 📊 Resumo da Aula

### O que aprendemos:

1. ✅ **Leitura de dados**: Schema inference vs schema explícito
2. ✅ **Transformações básicas**: `select`, `filter`, `withColumn`, `orderBy`
3. ✅ **Joins**: Como fazer joins entre múltiplos DataFrames
4. ✅ **Agregações**: `groupBy` e `agg` para criar métricas
5. ✅ **Conceitos fundamentais**: Transformations vs Actions, lazy evaluation

### Comparação: SQL vs PySpark DataFrame API

| Operação | SQL (Aula 06) | PySpark DataFrame API (Aula 07) |
|----------|---------------|----------------------------------|
| Leitura | `CREATE TABLE ... AS SELECT * FROM read_files(...)` | `spark.read.format("csv").load(...)` |
| Seleção | `SELECT col1, col2 FROM table` | `df.select("col1", "col2")` |
| Filtro | `WHERE condition` | `df.filter(condition)` ou `df.where(condition)` |
| Join | `INNER JOIN ... ON ...` | `df1.join(df2, on=..., how="inner")` |
| Agregação | `GROUP BY ... COUNT(*), AVG(...)` | `df.groupBy(...).agg(F.count("*"), F.avg(...))` |
| Nova coluna | `SELECT ..., new_col AS alias` | `df.withColumn("new_col", expression)` |

### Vantagens do PySpark DataFrame API:

- ✅ **Programação imperativa**: Mais flexível para lógica complexa
- ✅ **Reutilização**: Funções e classes podem ser reutilizadas
- ✅ **Testes**: Mais fácil de testar com unit tests
- ✅ **Integração**: Integra melhor com outras bibliotecas Python

### Quando usar SQL vs PySpark:

- **SQL**: Melhor para queries ad-hoc, análise exploratória, equipes mais SQL-oriented
- **PySpark**: Melhor para pipelines complexos, lógica reutilizável, integração com Python

---

## 🚀 Próximos Passos

Na próxima aula, vamos explorar:
- **Spark Runtime Architecture**: Jobs, Stages, Tasks
- **Otimizações**: Partitions, Broadcast Joins, Caching
- **Performance Tuning**: Como otimizar suas transformações

---
